# HVLP Global Gym Market Opportunity Model
**Click ▶ Run on the cell below — then follow the on-screen prompts.**

Scoring model: USA = 100 benchmark · 17 variables · 4 tiers

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HVLP Global Gym Market Scoring Tool — Single-Cell Colab Workflow
# ─────────────────────────────────────────────────────────────────────────────
import base64, io, os, zipfile, subprocess, sys, shutil
import ipywidgets as _w
from IPython.display import display, HTML, clear_output

# Inject CSS: force button label text to #FAEEFF on every .hvlp-btn widget
display(HTML(
    "<style>"
    "button.hvlp-btn, .hvlp-btn button {"
    "  color: #FAEEFF !important;"
    "  font-weight: 700 !important;"
    "}"
    "</style>"
))

# ─────────────────────────────────────────────────────────────────────────────
# Helper: create a branded button (#9600FA bg / #FAEEFF text)
# ─────────────────────────────────────────────────────────────────────────────
def _btn(label, width="240px"):
    b = _w.Button(
        description=label,
        layout=_w.Layout(width=width, height="44px"),
        _dom_classes=["hvlp-btn"],
    )
    b.style.button_color = "#9600FA"
    try:
        b.style.font_color = "#FAEEFF"
    except Exception:
        pass
    return b


# ─────────────────────────────────────────────────────────────────────────────
# Shared Output widgets — pre-built so they sit in the right place in the VBox
# ─────────────────────────────────────────────────────────────────────────────
_csv_out  = _w.Output()
_main_out = _w.Output()                                  # setup log + tool


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: everything after Continue is pressed
# ─────────────────────────────────────────────────────────────────────────────
def _run_workflow(_btn_event):
    _cont_btn.disabled = True

    with _main_out:

        # ── Install ──────────────────────────────────────────────────────────
        print("📦 Installing required packages…")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "requests", "pandas", "openpyxl", "numpy", "PyYAML", "ipywidgets"],
            check=True,
        )
        print("✅ Packages installed.")

        # ── Extract ──────────────────────────────────────────────────────────
        print("📂 Extracting model files…")
        _B64 = "..."
        _ZIP = base64.b64decode(_B64)
        with zipfile.ZipFile(io.BytesIO(_ZIP)) as _zf:
            _zf.extractall(".")
        os.makedirs("/content/output", exist_ok=True)
        os.makedirs("overrides", exist_ok=True)
        print("✅ Model files extracted.")

        # ── Load helpers ──────────────────────────────────────────────────────
        print("📚 Loading pipeline helpers…")
        sys.path.insert(0, ".")
        import pandas as pd
        import numpy as np
        import config as cfg
        from src.ingestor        import ingest_csv
        from src.calculator      import calculate_derived_metrics
        from src.fetcher         import fetch_all_external_data
        from src.override_loader import load_yaml_overrides, merge_overrides
        from src.normalizer      import normalize_all
        from src.weighter        import build_weight_matrix
        from src.scorer          import compute_scores
        from src.commentary      import generate_commentary
        from src.dashboard       import generate_dashboard
        from src.exporter        import export_excel
        print("✅ All helpers loaded — launching tool…")

    # ── Build interactive tool (imports available via closure) ────────────────
    PRELOADED = sorted(cfg.COUNTRY_ISO3_MAP.keys())
    CAT_LABELS = {
        "market_opportunity":   "Market Opportunity",
        "penetration_headroom": "Penetration Headroom",
        "operational_risk":     "Operational Risk",
        "cost_structure":       "Cost Structure",
        "demand_indicators":    "Demand Indicators",
    }
    VAR_LABELS = {
        "opportunity_usd_m":      "Opportunity ($M)",
        "potential_market_size":  "Potential Market Size ($M)",
        "gym_membership_cagr":    "Gym CAGR (%)",
        "penetration_headroom":   "Penetration Headroom",
        "concentration":          "Concentration (000s/gym)",
        "ease_of_doing_business": "Ease of Doing Business (GE.EST)",
        "political_stability":    "Political Stability",
        "inflation_rate":         "Inflation Rate (%)",
        "currency_volatility":    "Currency Volatility",
        "rule_of_law":            "Rule of Law",
        "financing_accessibility":"Financing Accessibility",
        "corporate_tax_rate":     "Corporate Tax Rate (%)",
        "labor_cost_index":       "Labour Cost Index",
        "real_estate_cost_index": "Real Estate Cost Index",
        "youth_population_pct":   "Youth / Working Age Population % (15–64)",
        "middle_class_pct":       "Middle Class (%)",
        "avg_gym_spend_pct_gdp":  "Avg Gym Spend as % GDP",
    }
    DASH_PATH = "/content/output/dashboard.html"
    _ts = {}   # tool state

    # ── CAGR overrides section ────────────────────────────────────────────────
    _cagr_w = {}
    for _c in PRELOADED:
        _lbl = (_c[:18] + ":") if len(_c) > 18 else (_c + ":")
        _cagr_w[_c] = _w.FloatText(
            value=0.0, description=_lbl,
            style={"description_width": "160px"},
            layout=_w.Layout(width="300px"),
        )
    _half = len(PRELOADED) // 2
    cagr_sec = _w.VBox([
        _w.HTML(
            "<div style='margin-top:10px'>"
            "<b style='color:#290241'>Gym Membership CAGR % — Optional Overrides</b><br>"
            "<span style='color:#6b7280;font-size:12px'>"
            "Leave at 0.0 to use default. Non-zero values are used as manual inputs."
            "</span></div>"
        ),
        _w.HBox([
            _w.VBox(list(_cagr_w.values())[:_half]),
            _w.VBox(list(_cagr_w.values())[_half:]),
        ]),
    ], layout=_w.Layout(
        border="1px solid #e2e8f0", padding="12px",
        margin="0 0 12px", border_radius="6px",
    ))

    # ── Pipeline ─────────────────────────────────────────────────────────────
    run_log = _w.Output(layout=_w.Layout(
        border="1px solid #ddd", padding="8px",
        overflow_y="auto", max_height="240px"))
    res_out = _w.Output()
    status  = _w.HTML("")

    def _cagr_overrides():
        return {c: {"gym_membership_cagr": float(w.value)}
                for c, w in _cagr_w.items() if float(w.value) != 0.0}

    def _pipeline(extra_rows=None):
        def _log(m):
            with run_log: print(m)
        _log("Loading CSV…")
        df = ingest_csv("input_data.csv", cfg.CSV_COLUMN_MAP)
        if extra_rows:
            df = pd.concat([df, pd.DataFrame(extra_rows)], ignore_index=True)
        countries = df["country"].tolist()
        _log(f"Derived metrics ({len(countries)} countries)…")
        df = calculate_derived_metrics(df, cfg.DUES_INCREASE_PCT)
        _log("Fetching external data (WB / OECD / TE)…")
        ext = fetch_all_external_data(
            countries=countries, country_iso3_map=cfg.COUNTRY_ISO3_MAP,
            wb_indicators=cfg.WB_INDICATORS,
            oecd_country_codes=cfg.OECD_COUNTRY_CODES,
            te_api_key=cfg.TRADING_ECONOMICS_API_KEY,
            cache_dir=cfg.CACHE_DIR, ttl_hours=cfg.CACHE_EXPIRY_HOURS,
        )
        _log("Merging overrides…")
        yaml_ov = load_yaml_overrides("overrides/manual_inputs.yaml")
        for c, vals in _cagr_overrides().items():
            yaml_ov.setdefault(c, {}).setdefault("gym_membership_cagr", vals["gym_membership_cagr"])
        scored = list(cfg.WEIGHTS.keys())
        df, audit = merge_overrides(df, ext, yaml_ov, scored)
        for c, vals in _cagr_overrides().items():
            if c in audit and audit[c].get("gym_membership_cagr") == "manual_yaml":
                audit[c]["gym_membership_cagr"] = "manual_input"
        _log("Normalising (USA-baseline)…")
        ndf = normalize_all(df, scored, cfg.INVERTED_VARIABLES, cfg.USA_BASELINE)
        _log("Building weight matrix…")
        avail = {r["country"]: {v: pd.notna(r.get(v)) for v in scored}
                 for _, r in df.iterrows()}
        wm = build_weight_matrix(
            countries=countries, availability_matrix=avail,
            base_weights=cfg.WEIGHTS, rule1_cfg=cfg.RULE1_MISSING_CAGR,
            rule2_cfg=cfg.RULE2_MISSING_CONCENTRATION,
            categories=cfg.VARIABLE_CATEGORIES,
        )
        _log("Computing scores…")
        sdf = compute_scores(
            normalized_df=ndf, weight_matrix=wm,
            categories=cfg.VARIABLE_CATEGORIES,
            tier_thresholds=cfg.TIER_THRESHOLDS,
            tier_labels=cfg.TIER_LABELS,
        )
        _log("Generating commentary…")
        comm = generate_commentary(
            scores_df=sdf, full_df=df, normalized_df=ndf,
            weight_matrix=wm, audit=audit,
            categories=cfg.VARIABLE_CATEGORIES,
            inverted_variables=cfg.INVERTED_VARIABLES,
        )
        _log("Writing outputs…")
        generate_dashboard(
            scores_df=sdf, full_df=df, normalized_df=ndf,
            weight_matrix=wm, audit=audit, commentary=comm,
            categories=cfg.VARIABLE_CATEGORIES, base_weights=cfg.WEIGHTS,
            tier_colors=cfg.TIER_COLORS,
            output_dir="/content/output", filename="dashboard.html",
        )
        export_excel(
            scores_df=sdf, full_df=df, normalized_df=ndf,
            weight_matrix=wm, audit=audit,
            categories=cfg.VARIABLE_CATEGORIES, base_weights=cfg.WEIGHTS,
            output_dir="/content/output", filename=cfg.EXCEL_FILENAME,
        )
        _log("✅ Done.")
        return sdf, df, ndf, wm, audit, comm

    # ── HTML builders ─────────────────────────────────────────────────────────
    TC = {"1":"#7c3aed","2":"#22c55e","3":"#3b82f6","4":"#f59e0b"}

    def _tn(tier):
        return "1" if "Tier 1" in tier else ("2" if "Tier 2" in tier else
               ("3" if "Tier 3" in tier else "4"))

    def _rankings_html(sdf):
        rows = ""
        for _, r in sdf.iterrows():
            tc = TC.get(_tn(str(r["tier"])), "#6b7280")
            rows += (
                f"<tr>"
                f"<td style='font-weight:700;color:#9600fa;padding:8px 12px'>#{int(r['rank'])}</td>"
                f"<td style='color:#290241;font-weight:600;padding:8px 12px'>{r['country']}</td>"
                f"<td style='font-weight:700;color:#9600fa;padding:8px 12px'>{float(r['composite_score']):.1f}</td>"
                f"<td style='padding:8px 12px'>"
                f"<span style='background:{tc};color:#fff;padding:3px 10px;"
                f"border-radius:20px;font-size:11px;font-weight:600'>{r['tier']}</span>"
                f"</td></tr>"
            )
        return (
            "<div style='font-family:sans-serif'>"
            "<div style='background:#290241;color:#FAEEFF;padding:16px 20px;border-radius:8px 8px 0 0'>"
            "<b style='font-size:16px'>HVLP Global Gym Market Rankings</b><br>"
            "<span style='font-size:11px;color:#d6b4f5'>USA = 100 benchmark | Scores &gt;100 = outperforming USA</span>"
            "</div>"
            "<table style='width:100%;border-collapse:collapse;background:#fff'>"
            "<thead><tr>"
            "<th style='background:#9600fa;color:#FAEEFF;padding:8px 12px;text-align:left'>Rank</th>"
            "<th style='background:#9600fa;color:#FAEEFF;padding:8px 12px;text-align:left'>Country</th>"
            "<th style='background:#9600fa;color:#FAEEFF;padding:8px 12px;text-align:left'>Score</th>"
            "<th style='background:#9600fa;color:#FAEEFF;padding:8px 12px;text-align:left'>Tier</th>"
            "</tr></thead><tbody>" + rows + "</tbody></table></div>"
        )

    def _scorecard_html(country, sdf, fdf, ndf, wm, audit):
        row = sdf[sdf["country"] == country]
        if row.empty:
            return f"<p>No data for: {country}</p>"
        row = row.iloc[0]
        rank = int(row["rank"]); score = float(row["composite_score"])
        tier = row["tier"]; total = len(sdf)
        ci = fdf[fdf["country"] == country].index
        if ci.empty:
            return f"<p>Not found: {country}</p>"
        pos = list(fdf.index).index(ci[0])
        nr = ndf.iloc[pos]
        fr = fdf[fdf["country"] == country].iloc[0]
        wts = wm.get(country, {})
        aud = audit.get(country, {})
        cbase = {cat: sum(cfg.WEIGHTS.get(v, 0) for v in vl)
                 for cat, vl in cfg.VARIABLE_CATEGORIES.items()}
        parts = [
            "<div style='font-family:sans-serif;max-width:920px'>",
            f"<div style='background:#290241;color:#FAEEFF;padding:16px 20px;border-radius:8px 8px 0 0'>"
            f"<b style='font-size:18px'>{country}</b></div>",
            "<div style='padding:12px 20px 4px;background:transparent'>",
            "<span style='color:#9600fa;font-size:14px'>Global Rank: </span>",
            f"<span style='color:#9600fa;font-weight:700;font-size:16px'>{rank} / {total}</span>",
            "<span style='color:#9600fa;font-size:14px'> &nbsp;|&nbsp; Score: </span>",
            f"<span style='color:#9600fa;font-weight:700;font-size:16px'>{score:.1f}</span>",
            f"<span style='color:#9600fa;font-size:14px'> &nbsp;|&nbsp; {tier}</span>",
            "</div>",
        ]
        for cat, vlist in cfg.VARIABLE_CATEGORIES.items():
            clbl = CAT_LABELS.get(cat, cat)
            cs   = float(row.get(f"contrib_{cat}", 0.0))
            cb   = cbase[cat] * 100
            parts += [
                "<div style='margin:12px 20px 0;border-left:4px solid #9600fa;"
                "background:#fff;border-radius:0 4px 4px 0;padding:8px 12px'>",
                f"<span style='color:#290241;font-weight:700;font-size:13px'>{clbl}</span>",
                f"<span style='color:#290241;font-size:12px'>"
                f" (base {cb:.0f}% → contribution: {cs:.2f} pts)</span></div>",
                "<div style='overflow-x:auto;margin:0 20px'>",
                "<table style='width:100%;border-collapse:collapse'>",
                "<thead><tr>",
                "<th style='background:#9600fa;color:#FAEEFF;padding:6px 10px;text-align:left;font-size:11px'>Variable</th>",
                "<th style='background:#9600fa;color:#FAEEFF;padding:6px 10px;text-align:right;font-size:11px'>Raw Value</th>",
                "<th style='background:#9600fa;color:#FAEEFF;padding:6px 10px;text-align:right;font-size:11px'>USA-Norm (×100)</th>",
                "<th style='background:#9600fa;color:#FAEEFF;padding:6px 10px;text-align:right;font-size:11px'>Wt%</th>",
                "<th style='background:#9600fa;color:#FAEEFF;padding:6px 10px;text-align:right;font-size:11px'>Contribution</th>",
                "<th style='background:#9600fa;color:#FAEEFF;padding:6px 10px;text-align:left;font-size:11px'>Source</th>",
                "</tr></thead><tbody>",
            ]
            for var in vlist:
                lbl   = VAR_LABELS.get(var, var)
                raw   = fr.get(var, float("nan"))
                norm  = nr.get(var, float("nan"))
                w     = wts.get(var, 0.0)
                src   = aud.get(var, "")
                import math
                raw_s  = f"{float(raw):,.2f}" if not math.isnan(float(raw) if raw is not None else float("nan")) else "—"
                norm_s = f"{float(norm)*100:.1f}" if not math.isnan(float(norm) if norm is not None else float("nan")) else "—"
                cont   = w * (float(norm) if norm_s != "—" else 0.0) * 100
                parts.append(
                    f"<tr style='border-bottom:1px solid #f1f5f9'>"
                    f"<td style='padding:6px 10px;color:#000'>{lbl}</td>"
                    f"<td style='padding:6px 10px;text-align:right;font-family:monospace;color:#000'>{raw_s}</td>"
                    f"<td style='padding:6px 10px;text-align:right;font-family:monospace;color:#000'>{norm_s}</td>"
                    f"<td style='padding:6px 10px;text-align:right;color:#000'>{w*100:.1f}%</td>"
                    f"<td style='padding:6px 10px;text-align:right;font-weight:600;color:#000'>{cont:.2f}pts</td>"
                    f"<td style='padding:6px 10px;color:#065f46;font-size:11px'>{src}</td>"
                    f"</tr>"
                )
            parts.append("</tbody></table></div>")
        parts += [
            "<div style='background:#290241;color:#d6b4f5;padding:12px 20px;"
            "margin-top:12px;border-radius:0 0 8px 8px;font-size:11px'>"
            "USA = 100 benchmark · Scores > 100 = outperforming USA</div></div>"
        ]
        return "".join(parts)

    def _show_dash():
        try:
            with open(DASH_PATH, encoding="utf-8") as f:
                content = f.read()
            with res_out:
                clear_output()
                display(HTML(content))
        except FileNotFoundError:
            with run_log:
                print("⚠️ Dashboard not found — did the pipeline complete?")

    # ── Tool buttons ──────────────────────────────────────────────────────────
    run_btn  = _btn("▶  Run All Countries",  width="200px")
    ctry_dd  = _w.Dropdown(options=PRELOADED, layout=_w.Layout(width="200px"))
    sc_btn   = _btn("📊 Scorecard",          width="130px")
    xl_btn   = _btn("⬇ Download Excel",     width="160px")
    dsh_btn  = _btn("🌐 Show Dashboard",     width="160px")
    add_btn  = _btn("➕ Add New Country",    width="170px")
    xl_btn.disabled  = True
    dsh_btn.disabled = True

    nc_name = _w.Text(placeholder="Country name",  layout=_w.Layout(width="180px"))
    nc_iso3 = _w.Text(placeholder="ISO3 e.g. VNM", layout=_w.Layout(width="120px"))
    nc_mkt  = _w.FloatText(description="Market $M:",    layout=_w.Layout(width="200px"))
    nc_cp   = _w.FloatText(description="Cur Pen %:", value=0.05, layout=_w.Layout(width="200px"))
    nc_fp   = _w.FloatText(description="Fut Pen %:", value=0.15, layout=_w.Layout(width="200px"))
    nc_pop  = _w.FloatText(description="Population M:", layout=_w.Layout(width="200px"))
    nc_con  = _w.FloatText(description="Conc. 000s:",   layout=_w.Layout(width="200px"))
    nc_gdp  = _w.FloatText(description="GDP/Capita $:", layout=_w.Layout(width="200px"))
    nc_cagr = _w.FloatText(description="CAGR (opt):", value=0.0, layout=_w.Layout(width="200px"))
    nc_sub  = _btn("Score This Country", width="180px")

    add_form = _w.VBox([
        _w.HTML("<b>New Country Inputs</b>"),
        _w.HBox([nc_name, nc_iso3]),
        _w.HBox([nc_mkt,  nc_cp]),
        _w.HBox([nc_fp,   nc_pop]),
        _w.HBox([nc_con,  nc_gdp]),
        _w.HBox([nc_cagr, nc_sub]),
    ], layout=_w.Layout(display="none", border="1px solid #ddd", padding="12px", margin="8px 0"))

    def on_run_all(_):
        xl_btn.disabled = True
        dsh_btn.disabled = True
        status.value = "<span style='color:#6b7280'>⏳ Running pipeline…</span>"
        with run_log: clear_output()
        with res_out:  clear_output()
        try:
            r = _pipeline()
            _ts.update(zip(["sdf","fdf","ndf","wm","audit","comm"], r))
            with res_out:
                clear_output()
                display(HTML(_rankings_html(_ts["sdf"])))
            _show_dash()
            xl_btn.disabled = False
            dsh_btn.disabled = False
            status.value = (
                f"<span style='color:#22c55e'>✅ {len(_ts['sdf'])} countries scored. "
                f"Dashboard displayed below.</span>"
            )
        except Exception as e:
            status.value = f"<span style='color:#ef4444'>❌ Error: {e}</span>"
            with run_log: print(f"ERROR: {e}")

    def on_scorecard(_):
        if not _ts.get("sdf") is not None:
            status.value = "<span style='color:#f59e0b'>⚠️ Run all countries first.</span>"
            return
        country = ctry_dd.value
        with res_out:
            clear_output()
            display(HTML(_scorecard_html(
                country, _ts["sdf"], _ts["fdf"], _ts["ndf"], _ts["wm"], _ts["audit"]
            )))
        status.value = f"<span style='color:#3b82f6'>📊 Scorecard: {country}</span>"

    def on_download(_):
        try:
            from google.colab import files as _cf
            _cf.download(f"/content/output/{cfg.EXCEL_FILENAME}")
        except Exception as e:
            with run_log: print(f"Download error: {e}")

    def on_show_dash(_):
        _show_dash()

    def on_add_toggle(_):
        add_form.layout.display = "none" if add_form.layout.display != "none" else "flex"

    def on_submit_new(_):
        cn = nc_name.value.strip()
        if not cn:
            status.value = "<span style='color:#ef4444'>❌ Enter a country name.</span>"
            return
        iso3 = nc_iso3.value.strip().upper()
        if iso3:
            cfg.COUNTRY_ISO3_MAP[cn] = iso3
            cfg.OECD_COUNTRY_CODES[cn] = iso3 if len(iso3) == 3 else None
        row = {
            "country": cn,
            "market_size_m":           float(nc_mkt.value),
            "current_penetration_pct": float(nc_cp.value),
            "future_penetration_pct":  float(nc_fp.value),
            "population_m":            float(nc_pop.value),
            "concentration":           float(nc_con.value),
            "gdp_per_capita":          float(nc_gdp.value),
        }
        if float(nc_cagr.value) != 0.0:
            row["gym_membership_cagr"] = float(nc_cagr.value)
        status.value = f"<span style='color:#6b7280'>⏳ Scoring {cn}…</span>"
        with run_log: clear_output()
        try:
            r = _pipeline(extra_rows=[row])
            _ts.update(zip(["sdf","fdf","ndf","wm","audit","comm"], r))
            with res_out:
                clear_output()
                display(HTML(_scorecard_html(
                    cn, _ts["sdf"], _ts["fdf"], _ts["ndf"], _ts["wm"], _ts["audit"]
                )))
            xl_btn.disabled = False
            dsh_btn.disabled = False
            status.value = f"<span style='color:#22c55e'>✅ {cn} scored and added.</span>"
        except Exception as e:
            status.value = f"<span style='color:#ef4444'>❌ Error: {e}</span>"
            with run_log: print(f"ERROR: {e}")

    run_btn.on_click(on_run_all)
    sc_btn.on_click(on_scorecard)
    xl_btn.on_click(on_download)
    dsh_btn.on_click(on_show_dash)
    add_btn.on_click(on_add_toggle)
    nc_sub.on_click(on_submit_new)

    # ── Render the tool ───────────────────────────────────────────────────────
    with _main_out:
        display(_w.VBox([
            _w.HTML(
                "<div style='background:#290241;color:#FAEEFF;padding:14px 20px;"
                "border-radius:8px;margin-bottom:10px'>"
                "<b style='font-size:16px'>HVLP Scoring Tool — Ready</b><br>"
                "<span style='font-size:12px;color:#d6b4f5'>"
                "USA = 100 benchmark · 17 variables · 4 tiers</span></div>"
            ),
            cagr_sec,
            _w.HBox([
                run_btn,
                _w.HTML("<span style='margin:0 6px;color:#9600fa'>│</span>"),
                ctry_dd, sc_btn,
                _w.HTML("<span style='margin:0 6px;color:#9600fa'>│</span>"),
                add_btn,
                _w.HTML("<span style='margin:0 6px;color:#9600fa'>│</span>"),
                xl_btn, dsh_btn,
            ]),
            status,
            add_form,
            run_log,
            res_out,
        ]))


# ─────────────────────────────────────────────────────────────────────────────
# Continue button (shown after Yes/No is clicked)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# Screen 3 — CONTINUE (hidden until YES/NO is clicked)
# ─────────────────────────────────────────────────────────────────────────────
_cont_btn = _btn("\u25b6  Continue", width="180px")
_cont_btn.on_click(_run_workflow)

_screen3 = _w.VBox(
    [
        _w.HTML(
            "<div style='margin:10px 0 6px'>"
            "<b style='color:#290241'>Click Continue to install packages and launch the tool.</b>"
            "</div>"
        ),
        _cont_btn,
        _main_out,
    ],
    layout=_w.Layout(display="none"),
)


# ─────────────────────────────────────────────────────────────────────────────
# Screen 2 — CSV Upload prompt (hidden until GO is clicked)
# ─────────────────────────────────────────────────────────────────────────────
_yes_btn = _btn("\u2713  Yes \u2014 upload my own CSV")
_no_btn  = _btn("\u2717  No \u2014 use preloaded data")


def _on_yes(_):
    _yes_btn.disabled = True
    _no_btn.disabled  = True
    from google.colab import files as _cf
    with _csv_out:
        print("\u2b06\ufe0f  Select your CSV file using the picker below\u2026")
        _up = _cf.upload()
        if _up:
            import shutil as _sh
            _fname = list(_up.keys())[0]
            _sh.copy(_fname, "input_data.csv")
            print(f"\u2705 Using uploaded file: {_fname}")
        else:
            print("\u2139\ufe0f  No file selected \u2014 using preloaded 20-country dataset.")
    _screen2.layout.display = "none"
    _screen3.layout.display = "flex"


def _on_no(_):
    _yes_btn.disabled = True
    _no_btn.disabled  = True
    with _csv_out:
        print("\u2139\ufe0f  Using preloaded 20-country dataset.")
    _screen2.layout.display = "none"
    _screen3.layout.display = "flex"


_yes_btn.on_click(_on_yes)
_no_btn.on_click(_on_no)

_screen2 = _w.VBox(
    [
        _w.HTML(
            "<div style='background:#290241;color:#FAEEFF;padding:16px 22px;"
            "border-radius:8px;margin-bottom:14px'>"
            "<b style='font-size:17px'>HVLP Global Gym Market Scoring Tool</b><br>"
            "<span style='font-size:12px;color:#d6b4f5'>"
            "USA = 100 benchmark &nbsp;\u00b7&nbsp; 17 variables &nbsp;\u00b7&nbsp; 4 tiers"
            "</span></div>"
        ),
        _w.HTML("<b style='color:#290241'>Do you want to upload your own CSV?</b>"),
        _w.HBox([_yes_btn, _no_btn], layout=_w.Layout(gap="12px", margin="8px 0")),
        _csv_out,
    ],
    layout=_w.Layout(display="none"),
)


# ─────────────────────────────────────────────────────────────────────────────
# Screen 1 — GO (shown on initial render)
# ─────────────────────────────────────────────────────────────────────────────
_go_btn = _btn("\u25b6  GO", width="180px")


def _on_go(_):
    _screen1.layout.display = "none"
    _screen2.layout.display = "flex"


_go_btn.on_click(_on_go)

_screen1 = _w.VBox(
    [
        _w.HTML(
            "<div style='background:#290241;color:#FAEEFF;padding:16px 22px;"
            "border-radius:8px;margin-bottom:14px'>"
            "<b style='font-size:17px'>HVLP Global Gym Market Scoring Tool</b><br>"
            "<span style='font-size:12px;color:#d6b4f5'>"
            "USA = 100 benchmark &nbsp;\u00b7&nbsp; 17 variables &nbsp;\u00b7&nbsp; 4 tiers"
            "</span></div>"
        ),
        _go_btn,
    ],
    layout=_w.Layout(display="flex"),
)


# ─────────────────────────────────────────────────────────────────────────────
# Initial render — the only display() call at module level
# ─────────────────────────────────────────────────────────────────────────────
display(_w.VBox([_screen1, _screen2, _screen3]))
